In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import modal

app = modal.App("scale-experiment")
volume = modal.Volume.from_name("test-volume", create_if_missing=True)

# The dashboard endpoint runs in a container and imports launchpad.status, so
# ship launchpad.py into its image (it isn't in the base image).
dashboard_image = modal.Image.debian_slim().pip_install("fastapi").add_local_python_source("launchpad")

RUNS = Path("/storage/runs")  # volume mount path, as seen inside the containers
COMMIT_EVERY = 5  # how often count_to flushes count.jsonl to the volume


def _utc():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


@app.function(volumes={"/storage": volume}, timeout=3600, scaledown_window=2, serialized=True)
def count_to(run_id, max_steps=20):
    """Count to max_steps, one {i, timestamp} row per step in count.jsonl.

    Writes its own config.json (max_steps) and modal_function_call_id.txt at
    startup -- so launching is just spawn(run_id, max_steps) from anywhere, and
    the dashboard can read progress and cancel it. Resumable: continues from the
    last committed line and keeps the original config on a resume.

    Single-owner guard: the call-id file is an ownership token. This container
    claims the folder by writing its own call id, then checks the file before
    doing any work (and again after each commit) -- if a newer spawn has
    overwritten it, this (older) container is no longer the owner and exits.
    Newest spawn wins; stale duplicates bow out, so two containers never write
    the same count.jsonl when a run gets (re)spawned twice.
    """
    rdir = RUNS / run_id
    rdir.mkdir(parents=True, exist_ok=True)

    cfg = rdir / "config.json"
    if not cfg.exists():  # first attempt fixes max_steps; resumes keep it
        cfg.write_text(json.dumps({"max_steps": max_steps}))

    cidfile = rdir / "modal_function_call_id.txt"
    my_id = modal.current_function_call_id()  # None only for a local .local() run
    if my_id:
        cidfile.write_text(my_id)  # claim ownership -- overwrites on every (re)spawn
    volume.commit()  # publish config + our claim before checking who won

    def i_own_it():
        """True if the call-id file still names this container. reload() pulls
        other containers' commits -- safe only right after our own commit, when
        there are no uncommitted local changes."""
        if not my_id:
            return True  # local run: no other container to contend with
        volume.reload()
        return cidfile.exists() and cidfile.read_text().strip() == my_id

    if not i_own_it():
        return  # a newer spawn already claimed this folder -- bow out before any work

    max_steps = json.loads(cfg.read_text())["max_steps"]  # config.json is the source of truth
    log = rdir / "count.jsonl"
    resume_from = json.loads(log.read_text().splitlines()[-1])["i"] if (log.exists() and log.stat().st_size) else 0

    for i in range(resume_from + 1, max_steps + 1):
        time.sleep(1)
        with open(log, "a") as f:
            f.write(json.dumps({"i": i, "timestamp": _utc()}) + "\n")
        if i % COMMIT_EVERY == 0:
            volume.commit()  # make progress visible to the dashboard container
            if not i_own_it():  # a newer spawn may have taken over since we started
                return
    volume.commit()  # flush the tail so `complete` shows up

In [2]:
# --- dashboard: one page, everything derived from the volume; polls itself smoothly ---
STATUS_STYLE = {
    "completed": ("Completed", "#1a7f37", "#dafbe1"),
    "in_progress": ("Running", "#9a6700", "#fff8c5"),
    "failed": ("Failed", "#cf222e", "#ffebe9"),
    "starting": ("Starting", "#57606a", "#f6f8fa"),
    "orphaned": ("Orphaned", "#57606a", "#f6f8fa"),
}

# A run that is neither alive (running/starting) nor complete can be resumed --
# count_to continues from the count.jsonl tail and keeps its config.
RESUMABLE = {"failed", "orphaned"}


def _derive(complete, has_rows, live):
    """launch_spec Sec.2 matrix: fold (progress, liveness) into one status."""
    if complete:
        return "completed"
    if live == "running":
        return "in_progress" if has_rows else "starting"
    if live in ("done", "failed", "crashed", "timed_out"):
        return "failed"
    return "orphaned"  # expired / unknown / no id


def _rows_html(rows):
    """Just the <tbody> contents -- returned on its own for the JS poll to swap in."""
    import html

    if not rows:
        return "<tr><td colspan='5'>No runs yet.</td></tr>"
    out = []
    for r in rows:
        label, fg, bg = STATUS_STYLE.get(r["state"], ("?", "#57606a", "#f6f8fa"))
        count = "—" if r["count"] is None else f"{r['count']:,}"
        total = "—" if r["max_steps"] is None else f"{r['max_steps']:,}"
        pct = 0.0 if not r["max_steps"] or r["count"] is None else min(100.0, 100 * r["count"] / r["max_steps"])
        name = html.escape(r["run_id"])
        if r["state"] == "in_progress" and r["call_id"]:
            action = (
                f"<a class='cancel' href='?action=cancel&run_id={name}' "
                f"onclick=\"return confirm('Cancel {name}?')\">Cancel</a>"
            )
        elif r["state"] in RESUMABLE:  # not alive, not complete -> resume from the tail
            # Debounce: one click can't fire twice (defends the double-click race).
            action = (
                f"<a class='resume' href='?action=resume&run_id={name}' "
                f"onclick=\"this.style.pointerEvents='none';this.textContent='Resuming…'\">Resume</a>"
            )
        else:
            action = "—"
        out.append(f"""
      <tr data-run='{name}'>
        <td class='mono'>{name}</td>
        <td><span class='badge' style='color:{fg};background:{bg}'>{label}</span></td>
        <td class='mono'>{count}</td>
        <td><div class='bar'><div class='fill' style='width:{pct:.0f}%'></div></div>
            <span class='mono small'>{count} / {total} ({pct:.0f}%)</span></td>
        <td>{action}</td>
      </tr>""")
    return "".join(out)


def _page(rows_html, n_runs, refresh_ms=5000):
    """Full page. The <tbody> is polled and swapped every refresh_ms without a
    full reload -- no flash, no scroll jump. Each poll re-runs the server scan."""
    return f"""<!doctype html><html><head><meta charset='utf-8'>
<title>Runs</title>
<style>
  :root {{ color-scheme: light dark; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; margin: 2rem; color: #1f2328; background: #fff; }}
  h1 {{ font-size: 1.25rem; margin: 0 0 1rem; }}
  table {{ border-collapse: collapse; width: 100%; font-size: .875rem; }}
  th, td {{ padding: .5rem .75rem; text-align: left; white-space: nowrap; }}
  thead th {{ border-bottom: 1px solid #d0d7de; color: #57606a; font-size: .72rem; text-transform: uppercase; letter-spacing: .02em; }}
  tbody tr {{ border-bottom: 1px solid #eaeef2; }}
  .mono {{ font-family: ui-monospace, SFMono-Regular, Menlo, monospace; }}
  .small {{ font-size: .75rem; color: #57606a; }}
  .badge {{ display: inline-block; padding: .12rem .5rem; border-radius: 999px; font-size: .72rem; font-weight: 600; }}
  .bar {{ display: inline-block; width: 120px; height: 6px; background: #eaeef2; border-radius: 3px; overflow: hidden; vertical-align: middle; margin-right: .5rem; }}
  .fill {{ height: 100%; background: #2da44e; transition: width .4s ease; }}
  a.cancel {{ color: #cf222e; }}
  a.resume {{ color: #0969da; }}
  @media (prefers-color-scheme: dark) {{
    body {{ background: #0d1117; color: #e6edf3; }}
    thead th {{ border-color: #30363d; color: #8b949e; }}
    tbody tr {{ border-color: #21262d; }}
    .bar {{ background: #21262d; }}
    .small {{ color: #8b949e; }}
    a.cancel {{ color: #ff7b72; }}
    a.resume {{ color: #4493f8; }}
  }}
</style></head>
<body>
<h1>Runs (<span id='count'>{n_runs}</span>)</h1>
<table>
  <thead><tr><th>Run</th><th>Status</th><th>Count</th><th>Progress</th><th>Action</th></tr></thead>
  <tbody id='runs'>{rows_html}</tbody>
</table>
<script>
const MS = {refresh_ms};
async function tick() {{
  try {{
    const r = await fetch('?partial=1', {{cache: 'no-store'}});
    if (!r.ok) return;
    document.getElementById('runs').innerHTML = await r.text();
    document.getElementById('count').textContent =
      document.querySelectorAll('#runs tr[data-run]').length;
  }} catch (e) {{}}
}}
setInterval(tick, MS);
</script>
</body></html>"""


@app.function(image=dashboard_image, volumes={"/storage": volume}, serialized=True)
@modal.fastapi_endpoint()
async def dashboard(action: str = "", run_id: str = "", partial: int = 0):
    """List runs from the volume with live count + progress. Running rows get a
    Cancel button; failed/orphaned rows get a Resume button. `?partial=1` returns
    just the <tbody> (what the page polls). Recomputed each request.

    Async endpoint -> use Modal's .aio interfaces (reload/cancel/spawn) instead of
    the blocking ones, or Modal warns about blocking calls in an async context.
    """
    import json as _json

    from fastapi.responses import HTMLResponse, RedirectResponse

    from launchpad import status  # shipped into dashboard_image

    runs_root = Path("/storage/runs")
    await volume.reload.aio()  # refresh our snapshot -- jobs write from other containers

    if action == "cancel" and run_id:
        cid = runs_root / run_id / "modal_function_call_id.txt"
        if cid.exists():
            await modal.FunctionCall.from_id(cid.read_text().strip()).cancel.aio()
        return RedirectResponse("./", status_code=303)

    if action == "resume" and run_id:
        # Guard: respawn only if the run isn't already alive, so hitting Resume
        # twice is a no-op the second time. The button also debounces clicks; a
        # truly simultaneous double-fire is the residual gap that count_to's
        # ownership check closes (fencing -- see the note under the job cell).
        # count_to continues from the count.jsonl tail and keeps its config.
        cidp = runs_root / run_id / "modal_function_call_id.txt"
        live = "unknown"
        if cidp.exists():
            _, live, _ = await status(modal.FunctionCall.from_id(cidp.read_text().strip()))
        if live != "running":
            await count_to.spawn.aio(run_id)
        return RedirectResponse("./", status_code=303)

    rows = []
    if runs_root.exists():
        for rdir in sorted(runs_root.iterdir(), reverse=True):  # most recent first
            if not rdir.is_dir():
                continue
            cfg = rdir / "config.json"
            max_steps_run = _json.loads(cfg.read_text())["max_steps"] if cfg.exists() else None
            log = rdir / "count.jsonl"
            count = (
                _json.loads(log.read_text().splitlines()[-1])["i"] if (log.exists() and log.stat().st_size) else None
            )
            complete = count is not None and max_steps_run is not None and count >= max_steps_run
            cidp = rdir / "modal_function_call_id.txt"
            call_id = cidp.read_text().strip() if cidp.exists() else None

            if complete:  # Sec.2 optimization: skip the Modal poll entirely
                state = "completed"
            elif call_id:
                _, live, _ = await status(modal.FunctionCall.from_id(call_id))
                state = _derive(complete, count is not None, live)
            else:
                state = _derive(complete, count is not None, "unknown")

            rows.append(
                {"run_id": rdir.name, "state": state, "count": count, "max_steps": max_steps_run, "call_id": call_id}
            )

    if partial:
        return HTMLResponse(_rows_html(rows))
    return HTMLResponse(_page(_rows_html(rows), len(rows)))

In [3]:
# --- deploy the app (job + endpoint) persistently, then print the dashboard URL ---
# app.deploy() registers everything server-side so the endpoint stays live after
# this notebook stops -- unlike `with app.run()`, which only lasts the block.
with modal.enable_output():
    app.deploy()

print("dashboard:", dashboard.get_web_url())

⠸ Creating objects.....
├── ⠋ Creating mount PythonPackage:launchpad: Finalizing index of 1 files
⠦ Creating objects......
├── 🔨 Created mount PythonPackage:launchpad
├── 🔨 Created function count_to.
⠧ Creating objects...
├── 🔨 Created mount PythonPackage:launchpad
├── 🔨 Created function count_to.
└── 🔨 Created web function dashboard => 
    https://ocanphys--scale-experiment-dashboard.modal.run
✓ Created objects.
├── 🔨 Created mount PythonPackage:launchpad
├── 🔨 Created function count_to.
└── 🔨 Created web function dashboard => 
    https://ocanphys--scale-experiment-dashboard.modal.run
✓ App deployed in 1.259s! 🎉

View Deployment: https://modal.com/apps/ocanphys/main/deployed/scale-experiment
dashboard: https://ocanphys--scale-experiment-dashboard.modal.run


In [4]:
# --- launch runs from the notebook (deploy cell above must have run) ---
# count_to writes its own config.json + call id, so launching is just spawn().
# Function.from_name targets the deployed app, so no `with app.run()` needed.
count_to_fn = modal.Function.from_name("scale-experiment", "count_to")

for name, steps in [("alpha", 20), ("beta", 40), ("gamma", 60)]:
    rid = f"{datetime.now():%Y%m%dT%H%M%S}_{name}"
    call = count_to_fn.spawn(rid, steps)
    print("spawned", rid, "->", call.object_id)

spawned 20260730T144859_alpha -> fc-01KYTG12RMMNZ8KH3XQ119996F
spawned 20260730T144859_beta -> fc-01KYTG12X4Y8SE5G2SAGG8ZC1W
spawned 20260730T144859_gamma -> fc-01KYTG131PY0V0K1B9W97TE1M8
